# CineMatch — Stage 3: Collaborative Filtering

In this stage, we implement **Collaborative Filtering** recommendation models. Unlike content-based filtering which relies on item descriptions, collaborative filtering recommends items based on the history of user ratings and user-item interactions.

### Core Concepts:
1. **User-Item Interaction Matrix**: A matrix where rows are users, columns are items, and cells contain user ratings. It is typically sparse because most users rate only a fraction of available items.
2. **Item-Item Collaborative Filtering (Neighborhood-based)**: Computes similarities between items based on their ratings vectors. A user's rating for an item is predicted as a similarity-weighted average of the user's ratings on similar items.
3. **Matrix Factorization (SVD)**: Decomposes the ratings matrix into low-rank matrices representing user and item latent features. Predicted ratings are computed by the dot product of these latent feature vectors.

Let's import libraries and load our train ratings and processed items.

In [1]:
import os
import sys
import pandas as pd
import numpy as np

# Ensure our src files are in the python path
sys.path.append(os.path.abspath(".."))

from src.recommendation.collaborative import CollaborativeRecommender

print("Recommender class successfully imported!")

Recommender class successfully imported!


## 1. Load Preprocessed Data

We load `movie_ratings_train.csv` and `book_ratings_train.csv` (the training splits), along with processed metadata files `movies_processed.csv` and `books_processed.csv` from the `data/processed` folder.

In [2]:
data_dir = "../data/processed"

# Movies
movie_train_df = pd.read_csv(os.path.join(data_dir, "movie_ratings_train.csv"))
movies_metadata = pd.read_csv(os.path.join(data_dir, "movies_processed.csv"))

# Books
book_train_df = pd.read_csv(os.path.join(data_dir, "book_ratings_train.csv"))
books_metadata = pd.read_csv(os.path.join(data_dir, "books_processed.csv"))

print(f"Loaded {len(movie_train_df)} movie ratings (train) and {len(book_train_df)} book ratings (train).")

Loaded 80896 movie ratings (train) and 186363 book ratings (train).


## 2. Movies: Fit Recommender and Compare Approaches

We train the collaborative models on movie training ratings.

In [3]:
movie_cf = CollaborativeRecommender(user_col='userId', item_col='movieId', rating_col='rating', num_factors=30)
movie_cf.fit(movie_train_df)

### 2.1 Predict Ratings
Let's predict ratings for a specific user and movie (e.g. user 1 on movie 31, which is *Dangerous Minds* if present, or other movies) and compare our models.

In [4]:
sample_user = movie_train_df.iloc[0]['userId']
sample_movie = movie_train_df.iloc[0]['movieId']
actual_rating = movie_train_df.iloc[0]['rating']

pred_ii = movie_cf.predict_rating_item_item(sample_user, sample_movie)
pred_svd = movie_cf.predict_rating_svd(sample_user, sample_movie)

print(f"Predicting for User {sample_user} on Movie {sample_movie}:")
print(f"Actual rating in training set: {actual_rating}")
print(f"Item-Item CF Prediction: {pred_ii:.2f}")
print(f"SVD Matrix Factorization Prediction: {pred_svd:.2f}")

Predicting for User 1.0 on Movie 804.0:
Actual rating in training set: 4.0
Item-Item CF Prediction: 4.39
SVD Matrix Factorization Prediction: 4.38


### 2.2 Recommendation Generation
Let's recommend unrated movies for a specific user (e.g., user 10) using both SVD and Item-Item models.

In [5]:
test_user_id = 10

recs_svd = movie_cf.recommend_collaborative(
    user_id=test_user_id, method='svd', top_k=5, 
    items_metadata_df=movies_metadata, title_col='title', genre_col='genres'
)

recs_ii = movie_cf.recommend_collaborative(
    user_id=test_user_id, method='item_item', top_k=5, 
    items_metadata_df=movies_metadata, title_col='title', genre_col='genres'
)

print(f"=== SVD RECOMMENDATIONS FOR USER {test_user_id} ===")
for r in recs_svd:
    print(f"Title: {r['title']} | Score: {r['score']:.4f} (Raw: {r['raw_rating']})")
print("\n")

print(f"=== ITEM-ITEM RECOMMENDATIONS FOR USER {test_user_id} ===")
for r in recs_ii:
    print(f"Title: {r['title']} | Score: {r['score']:.4f} (Raw: {r['raw_rating']})")

=== SVD RECOMMENDATIONS FOR USER 10 ===
Title: Men in Black (a.k.a. MIB) (1997) | Score: 0.6459 (Raw: 3.58)
Title: Mission: Impossible (1996) | Score: 0.6397 (Raw: 3.56)
Title: Wizard of Oz, The (1939) | Score: 0.6373 (Raw: 3.55)
Title: Honey, I Blew Up the Kid (1992) | Score: 0.6367 (Raw: 3.55)
Title: Crocodile Dundee (1986) | Score: 0.6332 (Raw: 3.53)


=== ITEM-ITEM RECOMMENDATIONS FOR USER 10 ===
Title: Quiet Earth, The (1985) | Score: 1.0000 (Raw: 5.0)
Title: Hard Ticket to Hawaii (1987) | Score: 0.9509 (Raw: 4.8)
Title: Pearl Jam Twenty (2011) | Score: 0.9509 (Raw: 4.8)
Title: Roommate, The (2011) | Score: 0.9509 (Raw: 4.8)
Title: Ring, The (1927) | Score: 0.9509 (Raw: 4.8)


## 3. Books: Fit Recommender and Compare Approaches

Let's run the collaborative recommendation logic on books.

In [6]:
book_cf = CollaborativeRecommender(user_col='user_id', item_col='book_id', rating_col='rating', num_factors=30)
book_cf.fit(book_train_df)

### 3.1 Predict Book Ratings

In [7]:
sample_user_b = book_train_df.iloc[0]['user_id']
sample_book_b = book_train_df.iloc[0]['book_id']
actual_rating_b = book_train_df.iloc[0]['rating']

pred_ii_b = book_cf.predict_rating_item_item(sample_user_b, sample_book_b)
pred_svd_b = book_cf.predict_rating_svd(sample_user_b, sample_book_b)

print(f"Predicting for User {sample_user_b} on Book {sample_book_b}:")
print(f"Actual rating in training set: {actual_rating_b}")
print(f"Item-Item CF Prediction: {pred_ii_b:.2f}")
print(f"SVD Matrix Factorization Prediction: {pred_svd_b:.2f}")

Predicting for User 1 on Book 45:
Actual rating in training set: 5
Item-Item CF Prediction: 3.29
SVD Matrix Factorization Prediction: 4.58


### 3.2 Book Recommendations

In [8]:
test_user_id_b = 5

recs_svd_b = book_cf.recommend_collaborative(
    user_id=test_user_id_b, method='svd', top_k=5, 
    items_metadata_df=books_metadata, title_col='title', genre_col='genres'
)

print(f"=== SVD BOOK RECOMMENDATIONS FOR USER {test_user_id_b} ===")
for r in recs_svd_b:
    print(f"Title: {r['title']} | Score: {r['score']:.4f} (Raw: {r['raw_rating']}) | Genres: {r['genres']}")

=== SVD BOOK RECOMMENDATIONS FOR USER 5 ===
Title: The Hunger Games (The Hunger Games, #1) | Score: 0.7931 (Raw: 4.17) | Genres: Classics|Romance|Fantasy|Mystery
Title: The Alchemist | Score: 0.7909 (Raw: 4.16) | Genres: Fiction
Title: Catching Fire (The Hunger Games, #2) | Score: 0.7873 (Raw: 4.15) | Genres: Fiction
Title: The Help | Score: 0.7857 (Raw: 4.14) | Genres: Fiction
Title: Harry Potter and the Sorcerer's Stone (Harry Potter, #1) | Score: 0.7837 (Raw: 4.13) | Genres: Sci-Fi|Romance|Fantasy|Mystery


Stage 3 Collaborative Filtering is fully complete! We have implemented Item-Item and SVD models that can predict ratings on a 1-5 scale and generate unrated recommendations for users. We are ready to proceed to Stage 4: Hybrid Recommender!